In [2]:
import optuna
# import mlflow
# import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
# import dagshub
import seaborn as sns
import pandas as pd
import warnings
from sklearn.model_selection import cross_val_score

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

# Paste the path you copied from the folder menu here
file_path = '/content/drive/MyDrive/sentiment_analysis/cleaned_data3.csv'

# Load the data into the 'df' variable
df = pd.read_csv(file_path)
df.dropna(inplace = True)
df["Sentiment"] = df["Sentiment"].astype("int8")

In [5]:
warnings.filterwarnings("ignore", category=UserWarning)

# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['Sentiment'] = df['Sentiment'].map({-1: 0, 0: 1, 1: 2})

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['Sentiment'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 5000   # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['CommentText'])
y = df['Sentiment']

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Function to optimize LightGBM hyperparameters
def objective(trial):
    # Define hyperparameters to be tuned
    param = {
        "objective": "multiclass",
        "num_class": 3,  # Assuming 3 categories (-1, 0, 1)
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "metric": "multi_logloss",
        "is_unbalance": True,
        "class_weight": "balanced"

    }

    # Define the LightGBM model with the trial parameters
    model = LGBMClassifier(**param, verbose = -1)

    # Perform cross-validation
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy')

    # Return the average score across folds
    return scores.mean()


# Create an Optuna study to optimize the hyperparameters
study = optuna.create_study(
    study_name="my_optimization_study",
    storage="sqlite:////content/drive/MyDrive/optuna_study.db", # Saves here!
    direction="maximize",
    load_if_exists=True )
study.optimize(objective, n_trials=50)

#  Extract the best hyperparameters
best_params = study.best_params
print(best_params)

[I 2026-07-10 04:04:55,316] Using an existing study with name 'my_optimization_study' instead of creating a new one.
[I 2026-07-10 04:30:20,667] Trial 29 finished with value: 0.6426017775531265 and parameters: {'learning_rate': 0.09270627074643983, 'n_estimators': 371, 'max_depth': 16}. Best is trial 25 with value: 0.6469245592974032.
[I 2026-07-10 04:54:10,513] Trial 30 finished with value: 0.6407747309450097 and parameters: {'learning_rate': 0.09060625568416962, 'n_estimators': 454, 'max_depth': 11}. Best is trial 25 with value: 0.6469245592974032.
[I 2026-07-10 05:18:52,734] Trial 31 finished with value: 0.6405707019649965 and parameters: {'learning_rate': 0.07576668763655507, 'n_estimators': 500, 'max_depth': 12}. Best is trial 25 with value: 0.6469245592974032.
[I 2026-07-10 05:43:58,389] Trial 32 finished with value: 0.6431440997451613 and parameters: {'learning_rate': 0.08601645200740372, 'n_estimators': 456, 'max_depth': 14}. Best is trial 25 with value: 0.6469245592974032.
[I 

In [ ]:
#  Extract the best hyperparameters
best_params = study.best_params
print(best_params)

In [ ]:

best_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    metric="multi_logloss",
    is_unbalance=True,
    class_weight="balanced",
    reg_alpha=0.1,   # L1 regularization
    reg_lambda=0.1,  # L2 regularization
    learning_rate=0.08,
    max_depth=20,
    n_estimators=367
)


# Fit the model on the resampled training data
best_model.fit(X_train, y_train)

In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 10.4 MB/s eta 0:00:00
